In [1]:
import pandas as pd
from datetime import datetime


def get_backtest_bounds(target_date: str, last_n_days: int):
    target_dt = pd.to_datetime(target_date)
    end_date_dt = target_dt - pd.Timedelta(days=1)
    start_date_dt = end_date_dt - pd.Timedelta(days=last_n_days)
    return start_date_dt.strftime('%Y-%m-%d'), end_date_dt.strftime('%Y-%m-%d')

def get_backtest_bounds(target_date: str, last_n_days: int):
    target_dt = pd.to_datetime(target_date)
    end_date_dt = target_dt - pd.offsets.BusinessDay(1)
    start_date_dt = end_date_dt - pd.offsets.BusinessDay(last_n_days)
    return start_date_dt.strftime('%Y-%m-%d'), end_date_dt.strftime('%Y-%m-%d')

# P.S. assume the data is queried
target_date = "2026-05-08"
last_n_days = 30
start_date, end_date = get_backtest_bounds(target_date, last_n_days)

In [2]:
# TO CHANGE
# use your save_dir
save_dir = "/Users/livardywufianto/Projects/Trading/data/minute_aggs"

# Load Data

In [3]:
import pandas as pd
from typing import List
ticker_list = [
    "AAPL",
    "MU",
]

import pandas as pd

def generate_csv_filepaths(start_date, end_date, save_dir):
    date_series = pd.date_range(start=start_date, end=end_date)    
    keys = []
    base_path = save_dir
    
    for dt in date_series:
        full_date = dt.strftime('%Y-%m-%d')
        key = f"{base_path}/{full_date}.csv.gz"
        keys.append(key)        
    return keys

def generate_csv_filepath(date_str, save_dir):
    dt = pd.to_datetime(date_str)
    base_path = save_dir
    full_date = dt.strftime('%Y-%m-%d')
    key = f"{base_path}/{full_date}.csv.gz"
    return key


def convert_timestamp_to_us_datetime(timestamp: int, unit="ms"):
    """
    Converts a millisecond timestamp to US/Eastern time.

    NYSE/NASDAQ are located at US/Eastern
    """
    return pd.to_datetime(
        timestamp, unit=unit, utc=True
    ).tz_convert('US/Eastern')      
    
def load_csv_by_chunking(
    filepath: str, ticker_list: List[str],
    chunk_size = 100000
):

    chunks = []
    for chunk in pd.read_csv(filepath, chunksize=chunk_size):
        filtered_chunk = chunk[chunk["ticker"].isin(ticker_list)]
        chunks.append(filtered_chunk)

    df = pd.concat(chunks)
    return df

In [4]:
csv_filepaths = generate_csv_filepaths(start_date, end_date, save_dir)

In [5]:
df_ls = []
for csv_filepath in csv_filepaths:
    try:
        # TO CHANGE
        # load_csv_by_chunking - use pd.read_csv()
        df_ls.append(
            load_csv_by_chunking(csv_filepath, ticker_list)
        )
    except: 
        pass
df = pd.concat(df_ls).reset_index(drop=True)

# TO CHANGE
# load_csv_by_chunking - use pd.read_csv()
target_df = load_csv_by_chunking(
    generate_csv_filepath(target_date, save_dir), ticker_list
)

In [6]:
import numpy as np

def aggregate_to_hour(df):
    df["us_datetime"] = df["window_start"].apply(
        lambda x: convert_timestamp_to_us_datetime(x, unit="ns"))
    df["us_date"] = df["us_datetime"].dt.date
    df["hour"] = df["us_datetime"].dt.hour
    agg_logic = {
        "open": "first",    # The first price of the hour
        "high": "max",      # The highest price in that hour
        "low": "min",       # The lowest price in that hour
        "close": "last",    # The final price of the hour
        "volume": "sum",    # Total volume traded in that hour
        "transactions": "sum" # Total count of trades in that hour
    }
    
    hour_df = df.groupby(
        ["ticker", "us_date", "hour"]
    ).agg(agg_logic).reset_index()
    return hour_df



# Volume Check

In [16]:
apply_log_norm = False
threshold = 2

hour_df = aggregate_to_hour(df)
target_hour_df = aggregate_to_hour(target_df)

if apply_log_norm:
    hour_df["volume"] = np.log1p(hour_df["volume"])
    target_hour_df["volume"] = np.log1p(target_hour_df["volume"])
    
volume_df = hour_df.groupby(["ticker", "hour"]).agg({
    "volume": ["mean", "std"]
})
volume_df.columns = [f"{col}_{stat}" for col, stat in volume_df.columns]
volume_df = volume_df.reset_index()

merge_df = pd.merge(target_hour_df, volume_df, on=["ticker", "hour"])
merge_df["z_score"] = (merge_df["volume"] - merge_df["volume_mean"]) / merge_df["volume_std"]
merge_df[merge_df["z_score"].abs() > threshold]

,ticker,us_date,hour,open,high,low,close,volume,transactions,volume_mean,volume_std,z_score
24,MU,2026-05-08,12,721.845,742.1499,721.8450,730.03,9.237463e+06,204921,4.427035e+06,1.756601e+06,2.738487
28,MU,2026-05-08,16,746.810,749.8800,720.6147,744.90,1.144602e+06,14112,5.164031e+05,3.068636e+05,2.047159


In [17]:
merge_df

,ticker,us_date,hour,open,high,low,close,volume,transactions,volume_mean,volume_std,z_score
0,AAPL,2026-05-08,4,287.8600,288.5300,287.6700,288.1000,6.352713e+04,2879,8.631653e+04,6.406281e+04,-0.355735
1,AAPL,2026-05-08,5,288.1700,288.3200,287.5000,288.1910,1.808169e+04,1112,2.809254e+04,2.745540e+04,-0.364623
2,AAPL,2026-05-08,6,288.2000,290.2000,287.7900,289.1450,3.644892e+04,1321,4.079263e+04,4.098658e+04,-0.105979
3,AAPL,2026-05-08,7,289.1000,291.1920,289.1000,290.8016,1.539493e+05,3334,1.438137e+05,1.444034e+05,0.070189
4,AAPL,2026-05-08,8,290.8000,291.6501,290.3810,290.3810,1.989918e+05,4095,1.662238e+05,1.419868e+05,0.230782
5,AAPL,2026-05-08,9,290.5200,294.5550,290.0000,293.9400,7.719237e+06,117016,5.712661e+06,2.926189e+06,0.685730
6,AAPL,2026-05-08,10,293.9100,294.7600,292.5200,293.5500,7.197754e+06,176584,5.946116e+06,2.508441e+06,0.498971
7,AAPL,2026-05-08,11,293.5309,294.2300,292.1801,292.3150,5.037282e+06,137862,4.020938e+06,1.441717e+06,0.704954
8,AAPL,2026-05-08,12,292.3000,292.9170,291.8800,292.1400,3.381484e+06,84930,3.576071e+06,1.661435e+06,-0.117120
9,AAPL,2026-05-08,13,292.1390,292.6300,291.7501,292.1350,3.047201e+06,64127,3.283812e+06,1.394331e+06,-0.169695


# Log Volume

In [14]:
apply_log_norm = True
threshold = 2

hour_df = aggregate_to_hour(df)
target_hour_df = aggregate_to_hour(target_df)
if apply_log_norm:
    hour_df["volume"] = np.log1p(hour_df["volume"])
    target_hour_df["volume"] = np.log1p(target_hour_df["volume"])
    
volume_df = hour_df.groupby(["ticker", "hour"]).agg({
    "volume": ["mean", "std"]
})
volume_df.columns = [f"{col}_{stat}" for col, stat in volume_df.columns]
volume_df = volume_df.reset_index()

merge_df = pd.merge(target_hour_df, volume_df, on=["ticker", "hour"])
merge_df["z_score"] = (merge_df["volume"] - merge_df["volume_mean"]) / merge_df["volume_std"]
merge_df[merge_df["z_score"].abs() > threshold]

,ticker,us_date,hour,open,high,low,close,volume,transactions,volume_mean,volume_std,z_score
24,MU,2026-05-08,12,721.845,742.1499,721.845,730.03,16.038778,204921,15.236942,0.360206,2.226048


In [15]:
merge_df

,ticker,us_date,hour,open,high,low,close,volume,transactions,volume_mean,volume_std,z_score
0,AAPL,2026-05-08,4,287.8600,288.5300,287.6700,288.1000,11.059238,2879,11.123478,0.735643,-0.087325
1,AAPL,2026-05-08,5,288.1700,288.3200,287.5000,288.1910,9.802710,1112,9.855462,0.903345,-0.058396
2,AAPL,2026-05-08,6,288.2000,290.2000,287.7900,289.1450,10.503695,1321,10.243392,0.842725,0.308882
3,AAPL,2026-05-08,7,289.1000,291.1920,289.1000,290.8016,11.944385,3334,11.602369,0.706198,0.484306
4,AAPL,2026-05-08,8,290.8000,291.6501,290.3810,290.3810,12.201024,4095,11.800312,0.644276,0.621958
5,AAPL,2026-05-08,9,290.5200,294.5550,290.0000,293.9400,15.859226,117016,15.472227,0.391976,0.987305
6,AAPL,2026-05-08,10,293.9100,294.7600,292.5200,293.5500,15.789280,176584,15.526748,0.374403,0.701199
7,AAPL,2026-05-08,11,293.5309,294.2300,292.1801,292.3150,15.432377,137862,15.152042,0.329145,0.851708
8,AAPL,2026-05-08,12,292.3000,292.9170,291.8800,292.1400,15.033826,84930,14.998995,0.422883,0.082364
9,AAPL,2026-05-08,13,292.1390,292.6300,291.7501,292.1350,14.929735,64127,14.927504,0.393916,0.005661
